In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
import scipy.stats as stats

In [2]:
# Read in and manipulate the percentage of blasts
blast = pd.read_excel('../data/All_haem_all_timepoints_Dec24.xlsx')
print(blast.shape)
blast_c1 = blast.loc[blast['Timepoint'].isin(['Screening', 'C1D1']), :]
print(blast_c1.shape)

blast_c1 = blast_c1.loc[blast_c1['Timepoint'].isin(['Screening', 'C1D1']), :]
blast_c1 = blast_c1.groupby(['Patient']).agg({
    'BMA_blasts': 'max',         # Maximum blast values
    'Platelets': 'min',          # Minimum platelet values
    'Neutrophils': 'min', 
    'Haemoglobin': 'min'# Minimum neutrophil values
})
blast_c1['Timepoint'] = 'C1D1'
blast_c1 = blast_c1.reset_index() 


blast_c1 = blast_c1.sort_values(by = ['Patient', 'Timepoint'])

# Note I have checked that the code below actually works by creating and visually inspecting a new column
# blast_c1['BMA_blasts'] = blast_c1.groupby('Patient')['BMA_blasts'].transform(lambda x: x.fillna(method='bfill'))
# blast_c1.head()

(454, 7)
(79, 7)


In [3]:
# Read in and then concatenate the C7D1 timepoint percent of blasts
blast_c7 = blast.loc[blast['Timepoint'].isin(['C7D1']) , :]
blasts = pd.concat([blast_c1, blast_c7], axis = 0)

print(blasts.shape)
blasts = blasts.loc[blasts['Timepoint'] != 'Screening', :]
print(blasts.shape)

blasts['Timepoint'] = blasts['Timepoint'].str.replace('C6D1', 'C7D1')

blasts = blasts.sort_values('Patient')
blasts.head(62)

(63, 7)
(63, 7)


,Patient,BMA_blasts,Platelets,Neutrophils,Haemoglobin,Timepoint,Days_since_C1D1
0,P01,12.0,42.0,0.4,109.0,C1D1,NaN
7,P01,5.0,47.0,1.1,105.0,C7D1,175.0
1,P02,15.0,36.0,0.8,80.0,C1D1,NaN
33,P02,7.0,46.0,1.0,76.0,C7D1,218.0
2,P03,6.0,89.0,2.9,96.0,C1D1,NaN
...,...,...,...,...,...,...,...
34,P35,5.0,57.0,0.4,87.0,C1D1,NaN
35,P36,13.0,22.0,0.5,96.0,C1D1,NaN
36,P37,30.0,206.0,2.6,79.0,C1D1,NaN
37,P38,11.0,91.0,0.4,73.0,C1D1,NaN


In [4]:
blasts.head()

,Patient,BMA_blasts,Platelets,Neutrophils,Haemoglobin,Timepoint,Days_since_C1D1
0,P01,12.0,42.0,0.4,109.0,C1D1,NaN
7,P01,5.0,47.0,1.1,105.0,C7D1,175.0
1,P02,15.0,36.0,0.8,80.0,C1D1,NaN
33,P02,7.0,46.0,1.0,76.0,C7D1,218.0
2,P03,6.0,89.0,2.9,96.0,C1D1,NaN


In [5]:
# Filter for samples with both a C1D1 and a C7D1 sample
# Please note that this removes patient 61201002 which doesn't have a blast measurement at C7D1
# Count occurrences of each value in the column
value_counts = blasts['Patient'].value_counts()


# Create a boolean mask for values occurring twice
mask = blasts['Patient'].isin(value_counts[value_counts == 2].index)

# Apply the mask to get filtered DataFrame
print(blasts.shape)
blasts = blasts[mask]
print(blasts.shape)

# Create a column for merging the dataframes
blasts['pid_tp'] = blasts['Patient'].astype(str) + '_' + blasts['Timepoint']

(63, 7)
(46, 7)


In [6]:
blasts.head()

,Patient,BMA_blasts,Platelets,Neutrophils,Haemoglobin,Timepoint,Days_since_C1D1,pid_tp
0,P01,12.0,42.0,0.4,109.0,C1D1,NaN,P01_C1D1
7,P01,5.0,47.0,1.1,105.0,C7D1,175.0,P01_C7D1
1,P02,15.0,36.0,0.8,80.0,C1D1,NaN,P02_C1D1
33,P02,7.0,46.0,1.0,76.0,C7D1,218.0,P02_C7D1
2,P03,6.0,89.0,2.9,96.0,C1D1,NaN,P03_C1D1


In [7]:
# Read in NULISA data and select blood samples
nulisa = pd.read_excel('../data/NULISA_for_upload.xlsx')



# Transpose the df so that it easier to work with 
nulisa.index = nulisa['targetName']
nulisa = nulisa.iloc[:, 1:]
nulisa = nulisa.T
nulisa

# select blood samples from the oral aza trial (these have _PB in the title and lack 'RN')
nulisa = nulisa.loc[nulisa.index.str.contains('_PB'), :]
nulisa.head()

nulisa.index = nulisa.index.str.replace('C1_D1', 'C1D1').str.replace('C7_D1', 'C7D1').str.replace('_PB', '')

nulisa = nulisa.loc[nulisa.index != 'P04_C12_D29', :]
nulisa.index

Index(['P08_C1D1', 'P08_C7D1', 'P24_C1D1', 'P24_C7D1', 'P12_C1D1', 'P12_C7D1',
       'P04_C1D1', 'P04_C7D1', 'P03_C1D1', 'P03_C7D1', 'P09_C1D1', 'P09_C7D1',
       'P10_C1D1', 'P10_C7D1', 'P16_C1D1', 'P16_C7D1', 'P02_C1D1', 'P02_C7D1',
       'P01_C1D1', 'P01_C7D1', 'P11_C1D1', 'P11_C7D1', 'P07_C1D1', 'P07_C7D1',
       'P17_C1D1', 'P17_C7D1', 'P18_C1D1', 'P18_C7D1', 'P22_C1D1', 'P22_C7D1',
       'P13_C1D1', 'P13_C7D1', 'P21_C1D1', 'P21_C7D1'],
      dtype='object')

In [8]:
nulisa.head()

targetName,AGER,AGRP,ANGPT1,ANGPT2,ANXA1,AREG,BDNF,BMP7,BST2,C1QA,...,TREM1,TREM2,VCAM1,VEGFA,VEGFC,VEGFD,VSNL1,VSTM1,WNT16,WNT7A
P08_C1D1,12.815814,11.599770,13.711416,12.817818,11.558501,9.357486,15.225674,11.779913,11.363372,12.427062,...,13.143437,12.017396,12.941936,15.142156,13.385485,7.592040,11.853413,14.683010,12.931866,8.744185
P08_C7D1,13.426360,13.281680,11.973600,12.801492,11.336655,9.546682,11.610879,12.108913,11.263604,12.575250,...,12.639026,12.237741,13.337745,14.208861,13.273854,7.704273,11.770859,11.012886,13.143388,8.900478
P24_C1D1,13.409686,12.211211,12.117352,14.589588,9.848232,10.399768,10.552040,12.523015,12.473277,13.134511,...,13.529944,12.498108,13.661783,14.344889,14.087294,7.527386,11.696018,11.279987,12.565671,8.744706
P24_C7D1,12.971496,13.604866,12.194923,15.528098,9.886488,10.453888,10.212944,12.194522,12.178838,13.449840,...,13.459456,12.441233,13.955447,14.797240,14.376489,7.352615,11.777130,10.086519,12.867314,9.052809
P12_C1D1,13.698245,12.930173,12.565802,13.068282,9.113249,10.701452,13.841984,12.688355,12.596144,14.120495,...,14.652326,13.940916,13.563734,14.163709,14.094700,8.116019,12.207011,11.916969,12.884989,9.620725


In [9]:
blasts.head()

,Patient,BMA_blasts,Platelets,Neutrophils,Haemoglobin,Timepoint,Days_since_C1D1,pid_tp
0,P01,12.0,42.0,0.4,109.0,C1D1,NaN,P01_C1D1
7,P01,5.0,47.0,1.1,105.0,C7D1,175.0,P01_C7D1
1,P02,15.0,36.0,0.8,80.0,C1D1,NaN,P02_C1D1
33,P02,7.0,46.0,1.0,76.0,C7D1,218.0,P02_C7D1
2,P03,6.0,89.0,2.9,96.0,C1D1,NaN,P03_C1D1


In [10]:
# Merge the nulisa data with the blast data
print(nulisa.shape)
nulisa = pd.merge(nulisa, blasts, left_index= True, right_on= 'pid_tp')
print(nulisa.shape)

(34, 247)
(32, 255)


In [13]:
nulisa.head()

,AGER,AGRP,ANGPT1,ANGPT2,ANXA1,AREG,BDNF,BMP7,BST2,C1QA,...,WNT16,WNT7A,Patient,BMA_blasts,Platelets,Neutrophils,Haemoglobin,Timepoint,Days_since_C1D1,pid_tp
7,12.815814,11.599770,13.711416,12.817818,11.558501,9.357486,15.225674,11.779913,11.363372,12.427062,...,12.931866,8.744185,P08,15.0,110.0,1.2,100.0,C1D1,NaN,P08_C1D1
140,13.426360,13.281680,11.973600,12.801492,11.336655,9.546682,11.610879,12.108913,11.263604,12.575250,...,13.143388,8.900478,P08,8.0,35.0,0.5,81.0,C7D1,214.0,P08_C7D1
11,13.698245,12.930173,12.565802,13.068282,9.113249,10.701452,13.841984,12.688355,12.596144,14.120495,...,12.884989,9.620725,P12,11.0,87.0,0.9,104.0,C1D1,NaN,P12_C1D1
206,13.972684,14.652388,13.903779,12.825494,9.158899,10.342438,15.547902,12.540153,10.867731,13.872861,...,13.044832,10.165778,P12,1.0,114.0,2.9,112.0,C7D1,168.0,P12_C7D1
3,13.202248,12.799180,11.184085,13.495764,9.587354,9.972133,6.338712,12.494508,10.816165,13.212585,...,12.684559,8.967549,P04,28.0,13.0,1.5,76.0,C1D1,NaN,P04_C1D1


In [15]:
# Create a df which details the difference between the C1D1 and C7D1 timepoint

nulisa_sorted = nulisa.sort_values(['Patient', 'Timepoint'])

# Identify numeric columns
numeric_cols = nulisa_sorted.select_dtypes(include=[np.number]).columns.tolist()
# Remove patient_id and timepoint from numeric columns if they are numeric
if 'PID' in numeric_cols:
    numeric_cols.remove('PID')
if 'Timepoint' in numeric_cols:
    numeric_cols.remove('Timepoint')

# # Group by patient_id and calculate the difference between first and second timepoint
result = []

for patient, group in nulisa_sorted.groupby('Patient'):
    if len(group) == 2:  # Only process patients with at least 2 timepoints
        first_row = group.iloc[0]
        second_row = group.iloc[1]

        print(second_row['Timepoint'])
#         # Start with copying non-numeric data from first timepoint
        diff_row = {'PID': patient}
        
        # For all non-numeric columns except patient_id and timepoint, keep first timepoint value
        for col in nulisa_sorted.columns:
            if col not in numeric_cols and col != 'PID' and col != 'Timepoint':
                diff_row[col] = first_row[col]
        
        # For numeric columns, calculate the difference
        for col in numeric_cols:
            diff_row[col] = second_row[col] - first_row[col]
            
        # Add a label to indicate this is a difference row
        diff_row['measurement_type'] = 'difference_t1_t2'
        
        result.append(diff_row)

# # Create the difference dataframe
diff_df = pd.DataFrame(result)

C7D1
C7D1
C7D1
C7D1
C7D1
C7D1
C7D1
C7D1
C7D1
C7D1
C7D1
C7D1
C7D1
C7D1
C7D1
C7D1


In [16]:
# For multiple correlations with a target variable using a spearman correlation
def calculate_correlations_with_pvalues(df, target_col, feature_cols):
    results = []
    
    for col in feature_cols:
        if col != target_col:
            corr, p_value = stats.spearmanr(df[target_col], df[col])
            results.append({
                'Feature': col,
                'Correlation': corr,
                'P-value': p_value
            })
    
    return pd.DataFrame(results).sort_values('Correlation', ascending=False)

In [17]:
for x, y in enumerate(nulisa.columns):
    print(x, y)

0 AGER
1 AGRP
2 ANGPT1
3 ANGPT2
4 ANXA1
5 AREG
6 BDNF
7 BMP7
8 BST2
9 C1QA
10 CALCA
11 CCL1
12 CCL11
13 CCL13
14 CCL14
15 CCL15
16 CCL16
17 CCL17
18 CCL19
19 CCL2
20 CCL20
21 CCL21
22 CCL22
23 CCL23
24 CCL24
25 CCL25
26 CCL26
27 CCL27
28 CCL28
29 CCL3
30 CCL4
31 CCL5
32 CCL7
33 CCL8
34 CD200
35 CD200R1
36 CD27
37 CD274
38 CD276
39 CD3E
40 CD4
41 CD40
42 CD40LG
43 CD46
44 CD70
45 CD80
46 CD83
47 CD93
48 CEACAM5
49 CHI3L1
50 CLEC4A
51 CNTF
52 CRP
53 CSF1
54 CSF1R
55 CSF2
56 CSF2RB
57 CSF3
58 CSF3R
59 CST7
60 CTF1
61 CTLA4
62 CTSS
63 CX3CL1
64 CXADR
65 CXCL1
66 CXCL10
67 CXCL11
68 CXCL12
69 CXCL13
70 CXCL14
71 CXCL16
72 CXCL2
73 CXCL3
74 CXCL5
75 CXCL6
76 CXCL8
77 CXCL9
78 EGF
79 EPO
80 FASLG
81 FGF19
82 FGF2
83 FGF21
84 FGF23
85 FLT1
86 FLT3LG
87 FLT4
88 FTH1
89 FURIN
90 GDF15
91 GDF2
92 GFAP
93 GRN
94 GZMA
95 GZMB
96 HAVCR1
97 HGF
98 HLA-DRA
99 ICAM1
100 ICOSLG
101 IFNA1; IFNA13
102 IFNA2
103 IFNB1
104 IFNG
105 IFNL1
106 IFNL2; IFNL3
107 IFNW1
108 IKBKG
109 IL10
110 IL10RB
111 IL11
112 

In [18]:
# Correlate the change in various molecules with the change in outcome
feature_columns = [col for col in nulisa.columns[:248] if col != 'target']

target = 'BMA_blasts'
correlation_results_blast_chng = calculate_correlations_with_pvalues(diff_df, target_col = target, feature_cols = feature_columns)
correlation_results_blast_chng
correlation_results_blast_chng.to_csv('blast_blood_correlation.csv')

In [19]:
files = glob.glob('../correlation_data/*.csv')

,Feature,Correlation,P-value
117,IL15,0.709443,0.002084
159,IL7R,0.697643,0.002659
202,SIRPA,0.650445,0.006367
125,IL17RA,0.648970,0.006528
212,TLR3,0.640121,0.007564
...,...,...,...
11,CCL1,-0.424781,0.100993
104,IFNG,-0.495577,0.050930
105,IFNL1,-0.505902,0.045572
112,IL12B,-0.628321,0.009144
